# Task 12 -- Directional-consistency tolerance calibration (`docs/open_questions.md` §8.1)

Empirically calibrates `physical_consistency_constraints.md`'s proposed
`delta_dir` tolerance -- the angular offset between wind-sea direction and
wind direction -- against real archive data, following §5's
percentile-calibration methodology already used for the other three
constraints. See `tasks/12-directional-consistency-tolerance.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 2, 1, tzinfo=timezone.utc)

X_raw = load_training_window(DATA_DIR, T_START, T_END, ["wad_come_from", "wsd", "was", "wsh"])

ocean_mask = (
    ~np.isnan(X_raw["wad_come_from"]).any(dim="time").values
    & ~np.isnan(X_raw["wsd"]).any(dim="time").values
)
print(f"{ocean_mask.mean():.1%} ocean")


## Empirical angular offset `|theta| = |angle(wsd) - angle(wad_come_from)|`

In [ ]:
theta_wind = X_raw["wad_come_from"].values  # degrees, coming_from convention
theta_wsea = X_raw["wsd"].values

offset = np.abs(((theta_wsea - theta_wind + 180.0) % 360.0) - 180.0)  # min circular distance
offset_ocean = offset[:, ocean_mask]
print(f"angular offset: median {np.nanmedian(offset_ocean):.1f} deg, "
      f"p90 {np.nanpercentile(offset_ocean, 90):.1f} deg, "
      f"p99 {np.nanpercentile(offset_ocean, 99):.1f} deg")


## Stratify by a wave-age proxy (young sea = larger tolerated offset, per §8.1's proposal)

In [ ]:
# Wave age proxy: wsh / was (larger => more developed sea, per the reduced
# ODE's fixed point wsh* ~ (kappa/c_ds) * was^2, correlated_variables.md §2.3).
# Not a calibrated wave-age measure -- illustrative stratification only.
wave_age_proxy = (X_raw["wsh"] / X_raw["was"].clip(min=0.1)).values

young = wave_age_proxy < np.nanpercentile(wave_age_proxy, 33)
developed = wave_age_proxy > np.nanpercentile(wave_age_proxy, 67)

for label, sel in [("young-proxy tercile", young), ("developed-proxy tercile", developed)]:
    vals = offset[sel & ocean_mask[None, :, :]]
    print(f"{label}: p90 offset {np.nanpercentile(vals, 90):.1f} deg "
          f"(n={np.sum(~np.isnan(vals))})")


## Decision-rule diagnostic

Following `physical_consistency_constraints.md` §5's own calibration
principle: pick `delta_dir` so a target percentile (95th/99th) of genuine
archive observations falls inside the tolerance, stratified by
season/region as that section recommends for the other constraints -- and,
per §8.1's specific proposal, compare the young-vs-developed split above to
decide whether the "unless young" exception is empirically justified.
Record the calibrated value(s) in `tasks/12-directional-consistency-tolerance.md` §4.